In [ ]:
include( "../geom.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/spd-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim

home, base = expfolder( "system" )

colormap1 = cgrad( :ice )
colormap2 = cgrad( [:black, :darkolivegreen, :yellowgreen, :white], [0.0, 0.20, 0.45, 0.90, 1.0] )

In [ ]:
# Hyper parameters.
N = 1024

# Create folder and extract critical noise.
folderdata = [
    createfolder( findfolder( N, ρlist[i], slist[j]; base=base, warn=false, crit=true ) )
    for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
slist[j]

In [ ]:
j = 15

# Unpack geometry variable for fixed speed.
θlist = vec.( readdlm.( folderdata[:,j].*"area-perimeter-scaling.txt" ) )

# Initialize data variables.
μℓdata = Vector{Vector{defFloat}}( undef, Nρ )
σℓdata = Vector{Vector{defFloat}}( undef, Nρ )
μAdata = Vector{Vector{defFloat}}( undef, Nρ )
σAdata = Vector{Vector{defFloat}}( undef, Nρ )

# Import cluster dynamical data.
for i ∈ 1:Nρ
    # Unpack the geometry of each cluster size.
    ℓdata = readdlm( folderdata[i,j]*"mean-convex-hull-perimeter.txt" )
    Adata = readdlm( folderdata[i,j]*"mean-convex-hull-area.txt" )

    # Unpack the mean and standard deviation separately.
    μℓdata[i] = ℓdata[:,1];  σℓdata[i] = ℓdata[:,2]
    μAdata[i] = Adata[:,1];  σAdata[i] = Adata[:,2]
end

In [ ]:
ρlist[i]

In [ ]:
# Choose focal parameter combination.
i = 21

# Unpack data for plotting.
θ = θlist[i][1:2]
μℓlist = μℓdata[i];  σℓlist = σℓdata[i]
μAlist = μAdata[i];  σAlist = σAdata[i]

# Unpack the scaling parameters across indeces.
βlist  = [θlist[i][1] for i ∈ 1:Nρ]
ℓ0list = [θlist[i][2] for i ∈ 1:Nρ]

# Cluster sizes to consider in plot.
jlist = 1:N
ιlist = 2:length( μℓlist )

# Plot dimensions.
ℓmin = floor( log10( minimum( μℓlist ) ) )
ℓmax = ceil(  log10( maximum( μℓlist ) ) );

In [ ]:
slist[i], ρlist[j]

In [ ]:
# Unpack geometry parameters for varying speed and density.
γdata = first.( vec.( readdlm.( folderdata.*"area-perimeter-scaling.txt" ) ), 2 )
βdata = first.( γdata );  ℓ0data = last.( γdata );

In [ ]:
# Plot the distribution of ℓ across i.
plt = plot(; size=(400,300), legend=:topleft )
plot!( plt; top_margin=-10pt, right_margin=10pt )

# Plot mean and fit.
scatter!( plt, ιlist, μℓlist; yerror=σℓlist,
    color=:gray, markersize=3.0, lw=1.5, fillcolor=:gray, alpha=1, label="" )
plot!( plt, jlist, polyloglaw.( jlist; β=θ ); color=:cornflowerblue, lw=2, label=L"ℓ_0 (\log(i+1))^β" )

xticks = 2.0.^(0:2:log2( N ))
plot!( plt; xlims=(1,N), xticks=(xticks, string.( round.( defInt, xticks ) )), xscale=:log10 )
plot!( plt; ylims=10.0.^[-2,2], yscale=:log10 )
plot!( plt; xlabel="cluster size, "*L"i", ylabel="convex hull perimeter, "*L"ℓ_i" )
plot!( plt; framestyle=:box, legendfontsize=9 )

# Plot the change in exponent for varying density and speed.
pltβ = plot!( plt; inset=(1, bbox( 0.35, 0.125, 0.225, 0.30, :bottom, :left )) )

heatmap!( plt[2], ρlist, slist, βdata; color=colormap1, colorbar=false )

plot!( plt[2]; xtickfontsize=8, ytickfontsize=8 )
plot!( plt[2]; xlims=ρlist[[1,Nρ]], xscale=:log10, ticks=ρlist[[1,Nρ]] )
plot!( plt[2]; ylims=slist[[1,Ns]], yscale=:log10, ticks=slist[[1,Ns]] )

annotate!( plt[2], 10^(-1.0), 10^(-2.5), text( L"ρ", "Computer Modern", 8, :center ) )
annotate!( plt[2], 10^(-2.5), 10^(-1.0), text( L"s", "Computer Modern", 8, :center; rotation=90 ) )

# Plot the change in scaling constant for changing density and speed.
pltℓ = plot!( plt; inset=(1, bbox( 0.725, 0.125, 0.225, 0.30, :bottom, :left )) )

heatmap!( plt[3], ρlist, slist, ℓ0data; color=colormap2, colorbar=false )

plot!( plt[3]; xlims=ρlist[[1,Nρ]], xscale=:log10, ticks=ρlist[[1,Nρ]] )
plot!( plt[3]; ylims=slist[[1,Ns]], yscale=:log10, ticks=slist[[1,Ns]] )
plot!( plt[3]; xtickfontsize=8, ytickfontsize=8 )

annotate!( plt[3], 10^(-1.0), 10^(-2.5), text( L"ρ", "Computer Modern", 8, :center ) )
annotate!( plt[3], 10^(-2.5), 10^(-1.0), text( L"s", "Computer Modern", 8, :center; rotation=90 ) )

# Plot a single colorbar for both heatmaps.
pltβ = plot!( plt; inset=(1, bbox( 0.35, 0.50, 0.225, 0.025, :bottom, :left )) )

xmap = 2.0:0.01:2.5
ymap = [0,1]
zmap = hcat( xmap, xmap )
heatmap!( plt[4], xmap, ymap, zmap'; color=colormap1, colorbar=false )

plot!( plt[4]; xticks=[2.0,2.5], xmirror=true )
plot!( plt[4]; yticks=[] )

annotate!( plt[4], 2.25, 6.0, text( L"β", "Computer Modern", 8, :center ) )
plot!( plt[4]; xtickfontsize=8, ytickfontsize=8 )

# Plot a single colorbar for both heatmaps.
pltβ = plot!( plt; inset=(1, bbox( 0.725, 0.50, 0.225, 0.025, :bottom, :left )) )

xmap = 4.0:0.1:8.0
ymap = [0,1]
zmap = hcat( xmap, xmap )
heatmap!( plt[5], xmap, ymap, zmap'; color=colormap2, colorbar=false )

xticks = [4.0,8.0]
plot!( plt[5]; xticks=(xticks,string.( round.( [4.0,8.0]; digits=1 ) )), xmirror=true )
plot!( plt[5]; yticks=[] )

annotate!( plt[5], 6.0, 6.0, text( L"ℓ_0", "Computer Modern", 8, :center ) )
plot!( plt[5]; xtickfontsize=8, ytickfontsize=8 )
plot!( plt; framestyle=:box )

# saveplot( plt, figurefolder*"geometry-area-perimeter_N-$(N).png"; background=:white, dpi=600 );